# Lesson 6b: Function Approximation - Practical

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement Tile Coding** - Production-ready feature construction
2. **Implement RBF Features** - Radial basis function approximation
3. **Implement Semi-Gradient SARSA** - Linear function approximation control
4. **Solve Mountain Car** - Classic continuous control problem
5. **Compare Feature Methods** - Tile coding vs RBF vs polynomial
6. **Analyze Convergence** - On-policy vs off-policy stability

This notebook provides production implementations for scaling RL to continuous domains.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import gymnasium as gym
from collections import defaultdict
from typing import Tuple, List, Callable
import itertools

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. Tile Coding Implementation

### Complete Tile Coder

In [ ]:
class TileCoder:
    """
    Tile Coding for continuous state spaces.
    
    Creates multiple offset grid tilings to provide:
    - Coarse coding (generalization)
    - Discrimination (precise values within tiles)
    """
    
    def __init__(self, 
                 state_low: np.ndarray,
                 state_high: np.ndarray,
                 num_tilings: int = 8,
                 tiles_per_dim: int = 8):
        """
        Args:
            state_low: Lower bounds of state space
            state_high: Upper bounds of state space
            num_tilings: Number of offset tilings
            tiles_per_dim: Number of tiles per dimension per tiling
        """
        self.state_low = np.array(state_low)
        self.state_high = np.array(state_high)
        self.num_tilings = num_tilings
        self.tiles_per_dim = tiles_per_dim
        self.state_dim = len(state_low)
        
        # Tile width for each dimension
        self.tile_width = (state_high - state_low) / tiles_per_dim
        
        # Create offsets for each tiling
        # Offset by fraction of tile width
        self.offsets = np.random.uniform(
            low=0, 
            high=self.tile_width,
            size=(num_tilings, self.state_dim)
        )
        
        # Total number of tiles per tiling
        self.tiles_per_tiling = tiles_per_dim ** self.state_dim
        
        # Total number of features
        self.num_features = num_tilings * self.tiles_per_tiling
        
    def get_tile_indices(self, state: np.ndarray) -> List[int]:
        """
        Get indices of active tiles for given state.
        
        Returns:
            List of num_tilings active tile indices
        """
        state = np.array(state)
        indices = []
        
        for tiling_idx in range(self.num_tilings):
            # Apply offset for this tiling
            offset_state = state - self.state_low - self.offsets[tiling_idx]
            
            # Find tile coordinates
            tile_coords = np.floor(offset_state / self.tile_width).astype(int)
            
            # Clip to valid range
            tile_coords = np.clip(tile_coords, 0, self.tiles_per_dim - 1)
            
            # Convert multi-dimensional coordinates to single index
            tile_idx = 0
            for dim in range(self.state_dim):
                tile_idx = tile_idx * self.tiles_per_dim + tile_coords[dim]
            
            # Global index across all tilings
            global_idx = tiling_idx * self.tiles_per_tiling + tile_idx
            indices.append(global_idx)
        
        return indices
    
    def encode(self, state: np.ndarray) -> np.ndarray:
        """
        Encode state as binary feature vector.
        
        Returns:
            Binary vector of length num_features
        """
        features = np.zeros(self.num_features)
        indices = self.get_tile_indices(state)
        features[indices] = 1
        return features
    
    def __repr__(self):
        return (f"TileCoder(dim={self.state_dim}, tilings={self.num_tilings}, "
                f"tiles_per_dim={self.tiles_per_dim}, features={self.num_features})")

print("✅ TileCoder implemented")

### Test Tile Coder

In [ ]:
# Test on 2D state space
tile_coder = TileCoder(
    state_low=[-1.2, -0.07],
    state_high=[0.6, 0.07],
    num_tilings=8,
    tiles_per_dim=8
)

print(tile_coder)
print(f"\nTotal features: {tile_coder.num_features}")

# Test encoding
state = np.array([0.0, 0.0])
features = tile_coder.encode(state)
print(f"\nState {state}:")
print(f"Active features: {np.sum(features)} (should be {tile_coder.num_tilings})")
print(f"Active indices: {np.where(features == 1)[0]}")

# Test nearby states
state2 = np.array([0.01, 0.01])
features2 = tile_coder.encode(state2)
overlap = np.sum(features * features2)
print(f"\nNearby state {state2}:")
print(f"Overlapping features: {overlap}/{tile_coder.num_tilings}")
print("(High overlap = good generalization)")

print("\n✅ Tile coder test complete")

## 2. RBF Features

### Radial Basis Function Implementation

In [ ]:
class RBFFeatures:
    """
    Radial Basis Function (RBF) features for continuous states.
    
    Uses Gaussian RBFs: φ_i(s) = exp(-||s - c_i||² / (2σ²))
    """
    
    def __init__(self,
                 state_low: np.ndarray,
                 state_high: np.ndarray,
                 num_rbfs_per_dim: int = 5,
                 sigma: float = 0.1):
        """
        Args:
            state_low: Lower bounds of state space
            state_high: Upper bounds of state space
            num_rbfs_per_dim: Number of RBF centers per dimension
            sigma: Width of RBFs (controls generalization)
        """
        self.state_low = np.array(state_low)
        self.state_high = np.array(state_high)
        self.state_dim = len(state_low)
        self.sigma = sigma
        
        # Create grid of RBF centers
        axes = [np.linspace(low, high, num_rbfs_per_dim) 
                for low, high in zip(state_low, state_high)]
        
        # All combinations of centers (grid)
        center_grid = np.array(list(itertools.product(*axes)))
        self.centers = center_grid
        
        self.num_features = len(self.centers)
        
    def encode(self, state: np.ndarray) -> np.ndarray:
        """
        Encode state using RBF features.
        
        Returns:
            Feature vector of length num_features
        """
        state = np.array(state)
        
        # Compute distance to each center
        distances = np.linalg.norm(self.centers - state, axis=1)
        
        # Gaussian RBF
        features = np.exp(-distances**2 / (2 * self.sigma**2))
        
        return features
    
    def __repr__(self):
        return (f"RBFFeatures(dim={self.state_dim}, "
                f"centers={len(self.centers)}, sigma={self.sigma})")

print("✅ RBFFeatures implemented")

### Visualize RBF Features

In [ ]:
# Create RBF features for 1D state
rbf_1d = RBFFeatures(
    state_low=[-1.0],
    state_high=[1.0],
    num_rbfs_per_dim=7,
    sigma=0.2
)

# Plot RBF activations
states = np.linspace(-1.0, 1.0, 100)
fig, ax = plt.subplots(figsize=(12, 5))

for i in range(rbf_1d.num_features):
    activations = [rbf_1d.encode([s])[i] for s in states]
    ax.plot(states, activations, label=f'RBF {i}')

# Mark centers
ax.scatter(rbf_1d.centers.flatten(), [0]*len(rbf_1d.centers), 
           s=100, c='red', marker='x', zorder=10, label='Centers')

ax.set_xlabel('State')
ax.set_ylabel('Feature Activation')
ax.set_title('RBF Feature Activations (σ=0.2)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ RBF visualization complete")

## 3. Semi-Gradient SARSA

### Implementation with Linear Function Approximation

In [ ]:
class SemiGradientSARSA:
    """
    Semi-Gradient SARSA with Linear Function Approximation.
    
    Q(s,a) = w^T x(s,a)
    
    Update: w ← w + α[R + γQ(S',A') - Q(S,A)] x(S,A)
    """
    
    def __init__(self,
                 feature_encoder: Callable,
                 num_features: int,
                 num_actions: int,
                 alpha: float = 0.01,
                 gamma: float = 0.99,
                 epsilon: float = 0.1):
        """
        Args:
            feature_encoder: Function to encode state → features
            num_features: Number of features
            num_actions: Number of actions
            alpha: Learning rate
            gamma: Discount factor
            epsilon: Exploration rate
        """
        self.feature_encoder = feature_encoder
        self.num_features = num_features
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Separate weights for each action
        self.w = np.zeros((num_actions, num_features))
        
    def q_value(self, state: np.ndarray, action: int) -> float:
        """
        Compute Q(s,a) = w_a^T x(s)
        """
        features = self.feature_encoder(state)
        return np.dot(self.w[action], features)
    
    def get_action(self, state: np.ndarray) -> int:
        """
        ε-greedy action selection.
        """
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            # Greedy action
            q_values = [self.q_value(state, a) for a in range(self.num_actions)]
            return np.argmax(q_values)
    
    def update(self, state, action, reward, next_state, next_action, done):
        """
        Semi-gradient SARSA update.
        
        w ← w + α[R + γQ(S',A') - Q(S,A)] x(S)
        """
        features = self.feature_encoder(state)
        
        # Current Q-value
        q_current = np.dot(self.w[action], features)
        
        # Target
        if done:
            target = reward
        else:
            q_next = self.q_value(next_state, next_action)
            target = reward + self.gamma * q_next
        
        # TD error
        td_error = target - q_current
        
        # Update weights for this action
        self.w[action] += self.alpha * td_error * features
    
    def train_episode(self, env) -> float:
        """
        Train on one episode.
        
        Returns:
            Total episode reward
        """
        state, _ = env.reset()
        action = self.get_action(state)
        total_reward = 0
        
        for step in range(1000):
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
            
            if done:
                # Terminal update
                self.update(state, action, reward, next_state, None, True)
                break
            else:
                next_action = self.get_action(next_state)
                self.update(state, action, reward, next_state, next_action, False)
                
                state = next_state
                action = next_action
        
        return total_reward
    
    def train(self, env, n_episodes: int = 1000, verbose: bool = True) -> List[float]:
        """
        Train agent over multiple episodes.
        """
        rewards = []
        
        for episode in range(n_episodes):
            ep_reward = self.train_episode(env)
            rewards.append(ep_reward)
            
            if verbose and (episode + 1) % 100 == 0:
                avg_reward = np.mean(rewards[-100:])
                print(f"Episode {episode + 1}/{n_episodes}, Avg Reward: {avg_reward:.2f}")
        
        return rewards

print("✅ Semi-Gradient SARSA implemented")

## 4. Mountain Car with Tile Coding

### Environment Setup

In [ ]:
# Create Mountain Car environment
env = gym.make('MountainCar-v0')

print("Mountain Car Environment:")
print(f"State space: {env.observation_space}")
print(f"  Position: [{env.observation_space.low[0]:.2f}, {env.observation_space.high[0]:.2f}]")
print(f"  Velocity: [{env.observation_space.low[1]:.2f}, {env.observation_space.high[1]:.2f}]")
print(f"Action space: {env.action_space} (0=left, 1=nothing, 2=right)")
print(f"Goal: Reach position >= 0.5")
print(f"Challenge: Not enough power to go directly - must build momentum")

### Train with Tile Coding

In [ ]:
# Create tile coder
tile_coder = TileCoder(
    state_low=env.observation_space.low,
    state_high=env.observation_space.high,
    num_tilings=8,
    tiles_per_dim=8
)

print(f"Using {tile_coder}")
print(f"\nTraining Semi-Gradient SARSA with Tile Coding...\n")

# Create agent
agent_tile = SemiGradientSARSA(
    feature_encoder=tile_coder.encode,
    num_features=tile_coder.num_features,
    num_actions=env.action_space.n,
    alpha=0.1 / tile_coder.num_tilings,  # Normalize by number of active features
    gamma=0.99,
    epsilon=0.1
)

# Train
rewards_tile = agent_tile.train(env, n_episodes=500, verbose=True)

print("\n✅ Training complete")

In [ ]:
# Plot learning curve
fig, ax = plt.subplots(figsize=(12, 5))

# Smooth with moving average
window = 50
smoothed = np.convolve(rewards_tile, np.ones(window)/window, mode='valid')

ax.plot(smoothed, linewidth=2, label='Tile Coding')
ax.axhline(y=-110, color='green', linestyle='--', label='Near-optimal (-110)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (50-episode moving average)')
ax.set_title('Mountain Car: Learning with Tile Coding')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final performance (last 100 episodes): {np.mean(rewards_tile[-100:]):.2f}")

## 5. Compare Feature Methods

### Train with RBF Features

In [ ]:
# Create RBF features
rbf_features = RBFFeatures(
    state_low=env.observation_space.low,
    state_high=env.observation_space.high,
    num_rbfs_per_dim=8,
    sigma=0.15
)

print(f"Using {rbf_features}")
print(f"\nTraining Semi-Gradient SARSA with RBF Features...\n")

# Create agent
agent_rbf = SemiGradientSARSA(
    feature_encoder=rbf_features.encode,
    num_features=rbf_features.num_features,
    num_actions=env.action_space.n,
    alpha=0.01,
    gamma=0.99,
    epsilon=0.1
)

# Train
rewards_rbf = agent_rbf.train(env, n_episodes=500, verbose=True)

print("\n✅ RBF training complete")

### Train with Polynomial Features

In [ ]:
def polynomial_features(state, order=3):
    """
    Create polynomial features up to given order.
    
    For 2D state [x, y]:
    Order 1: [1, x, y]
    Order 2: [1, x, y, x², xy, y²]
    Order 3: [1, x, y, x², xy, y², x³, x²y, xy², y³]
    """
    features = [1.0]  # Bias term
    
    # Normalize state to [-1, 1]
    norm_state = 2 * (state - env.observation_space.low) / \
                 (env.observation_space.high - env.observation_space.low) - 1
    
    # Generate all polynomial combinations
    for deg in range(1, order + 1):
        for powers in itertools.combinations_with_replacement(range(len(state)), deg):
            feature = 1.0
            for p in powers:
                feature *= norm_state[p]
            features.append(feature)
    
    return np.array(features)

# Count features
sample_state = env.observation_space.sample()
poly_features = polynomial_features(sample_state, order=3)
num_poly_features = len(poly_features)

print(f"Polynomial features (order 3): {num_poly_features} features")
print(f"\nTraining Semi-Gradient SARSA with Polynomial Features...\n")

# Create agent
agent_poly = SemiGradientSARSA(
    feature_encoder=lambda s: polynomial_features(s, order=3),
    num_features=num_poly_features,
    num_actions=env.action_space.n,
    alpha=0.001,  # Lower learning rate for polynomial
    gamma=0.99,
    epsilon=0.1
)

# Train
rewards_poly = agent_poly.train(env, n_episodes=500, verbose=True)

print("\n✅ Polynomial training complete")

### Compare All Methods

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Learning curves
window = 50
methods = [
    ('Tile Coding', rewards_tile, tile_coder.num_features),
    ('RBF', rewards_rbf, rbf_features.num_features),
    ('Polynomial', rewards_poly, num_poly_features)
]

for name, rewards, num_feat in methods:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(smoothed, linewidth=2, label=f'{name} ({num_feat} features)')

ax1.axhline(y=-110, color='green', linestyle='--', alpha=0.5, label='Near-optimal')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward (50-episode moving average)')
ax1.set_title('Mountain Car: Feature Method Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Final performance bar chart
names = [name for name, _, _ in methods]
final_perfs = [np.mean(rewards[-100:]) for _, rewards, _ in methods]
colors = sns.color_palette("husl", len(methods))

bars = ax2.bar(names, final_perfs, color=colors)
ax2.set_ylabel('Final Performance (last 100 episodes)')
ax2.set_title('Final Performance Comparison')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, final_perfs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("\nFinal Performance Summary:")
print("=" * 50)
for name, rewards, num_feat in methods:
    final = np.mean(rewards[-100:])
    print(f"{name:15s}: {final:7.2f} ({num_feat} features)")
print("=" * 50)

## 6. Visualize Learned Value Function

### Plot Q-values for Best Agent

In [ ]:
# Use tile coding agent (typically best performer)
agent = agent_tile

# Create grid of states
positions = np.linspace(env.observation_space.low[0], env.observation_space.high[0], 50)
velocities = np.linspace(env.observation_space.low[1], env.observation_space.high[1], 50)

# Compute value function V(s) = max_a Q(s,a)
V = np.zeros((len(velocities), len(positions)))
for i, vel in enumerate(velocities):
    for j, pos in enumerate(positions):
        state = np.array([pos, vel])
        q_values = [agent.q_value(state, a) for a in range(env.action_space.n)]
        V[i, j] = np.max(q_values)

# 3D surface plot
fig = plt.figure(figsize=(14, 10))

# Value function surface
ax1 = fig.add_subplot(221, projection='3d')
X, Y = np.meshgrid(positions, velocities)
surf = ax1.plot_surface(X, Y, V, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Position')
ax1.set_ylabel('Velocity')
ax1.set_zlabel('Value')
ax1.set_title('Learned Value Function V(s)')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot
ax2 = fig.add_subplot(222)
contour = ax2.contourf(X, Y, V, levels=20, cmap='viridis')
ax2.axvline(x=0.5, color='red', linestyle='--', label='Goal')
ax2.set_xlabel('Position')
ax2.set_ylabel('Velocity')
ax2.set_title('Value Function Contours')
ax2.legend()
fig.colorbar(contour, ax=ax2)

# Optimal policy visualization
ax3 = fig.add_subplot(223)
policy = np.zeros((len(velocities), len(positions)))
for i, vel in enumerate(velocities):
    for j, pos in enumerate(positions):
        state = np.array([pos, vel])
        q_values = [agent.q_value(state, a) for a in range(env.action_space.n)]
        policy[i, j] = np.argmax(q_values)

policy_plot = ax3.contourf(X, Y, policy, levels=[-0.5, 0.5, 1.5, 2.5], 
                           colors=['blue', 'gray', 'red'], alpha=0.6)
ax3.axvline(x=0.5, color='green', linestyle='--', linewidth=2, label='Goal')
ax3.set_xlabel('Position')
ax3.set_ylabel('Velocity')
ax3.set_title('Learned Policy (Blue=Left, Gray=Nothing, Red=Right)')
ax3.legend()

# Sample trajectory
ax4 = fig.add_subplot(224)
state, _ = env.reset()
trajectory = [state]
agent.epsilon = 0  # Greedy policy

for _ in range(200):
    action = agent.get_action(state)
    next_state, reward, terminated, truncated, _ = env.step(action)
    trajectory.append(next_state)
    if terminated or truncated:
        break
    state = next_state

trajectory = np.array(trajectory)
ax4.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=2, alpha=0.7)
ax4.plot(trajectory[0, 0], trajectory[0, 1], 'go', markersize=10, label='Start')
ax4.plot(trajectory[-1, 0], trajectory[-1, 1], 'ro', markersize=10, label='End')
ax4.axvline(x=0.5, color='green', linestyle='--', label='Goal')
ax4.set_xlabel('Position')
ax4.set_ylabel('Velocity')
ax4.set_title(f'Sample Trajectory ({len(trajectory)} steps)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ Trajectory reached goal in {len(trajectory)} steps")

## Summary and Key Insights

### Implementations Complete

✅ **Tile Coding** - Production-ready coarse coding  
✅ **RBF Features** - Smooth radial basis functions  
✅ **Polynomial Features** - Global approximation  
✅ **Semi-Gradient SARSA** - Linear function approximation control  
✅ **Mountain Car Solution** - Classic continuous control benchmark  

### Performance Comparison

**Tile Coding**:
- ✅ Best performance (typically -110 to -120)
- ✅ Fast learning
- ✅ Efficient (sparse features)
- ✅ Good generalization + discrimination

**RBF Features**:
- ~ Moderate performance
- ✅ Smooth value function
- ❌ All features active (slower)
- ~ Requires tuning σ

**Polynomial Features**:
- ~ Can work well with tuning
- ❌ Global approximation (may miss local details)
- ❌ Can be unstable
- ~ Requires careful learning rate

### Key Findings

1. **Feature choice matters**: Tile coding clearly superior for Mountain Car
2. **Learning rate critical**: Must normalize for number of active features
3. **Generalization vs discrimination**: Tile coding balances both
4. **Convergence**: On-policy SARSA stable with all feature types
5. **Scalability**: Tile coding scales better to higher dimensions

### Practical Recommendations

**Default choice**: Tile coding
- 8 tilings × 8 tiles/dim works well
- Normalize learning rate: α / num_tilings

**For smooth functions**: RBF
- Tune σ carefully
- Watch computational cost

**For simple problems**: Polynomial
- Keep order low (2-3)
- Normalize state space

### What We Learned

1. Function approximation enables continuous RL
2. Feature engineering is critical (until deep RL)
3. Linear methods work surprisingly well
4. Semi-gradient methods converge (on-policy)
5. Tile coding is the workhorse of classic RL

---

**Lesson 6b Complete!** 🎉

You now have production implementations of function approximation methods, ready to scale RL to continuous domains. Next: Deep RL!